In [8]:
import numpy as np
import torch

EMBED_PATH   = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/模型文件/embeddings_best.pt"
BASIS_DIR    = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/pattern_basis"
ALPHA_DIR    = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/α与quality"
SAVE_DIR     = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/pattern_influence"
UID_PATH     = "/Users/yubinhao/ml-1m/Amazon Review/MF/实验用户id/sampled_uids.npy"
TOP_K        = 8
LAMBDA_PROBE = 1.0

# ── 加载固定用户列表 ───────────────────────────────────────────────────────────
sampled_uids = np.load(UID_PATH).tolist()
print(f"已加载 {len(sampled_uids)} 名用户，首5个uid: {sampled_uids[:5]}")

# ── 1. 加载所需数据 ────────────────────────────────────────────────────────────
print("[1] 加载数据...")
ckpt     = torch.load(EMBED_PATH, map_location="cpu")
user_emb = ckpt["user_embeddings"].numpy()   # (n_users, d)
item_emb = ckpt["item_embeddings"].numpy()   # (n_items, d)

basis_ckpt        = torch.load(f"{BASIS_DIR}/pattern_basis.pt", map_location="cpu")
pattern_basis     = basis_ckpt["pattern_basis"].numpy()   # (K_total, d)

sets_ckpt         = torch.load(f"{BASIS_DIR}/pattern_item_sets.pt", map_location="cpu")
pattern_item_sets = sets_ckpt["pattern_item_sets"]        # dict: k → list

hq_ckpt    = torch.load(f"{ALPHA_DIR}/high_quality_pearson.pt", map_location="cpu")
hq_patterns = hq_ckpt["H"]                               # e.g. [1,2,3,4,5,6,7,9]
K = len(hq_patterns)
print(f"  高质量 pattern 数: {K}，物品数: {item_emb.shape[0]}")

# ── 2. 计算 influence 矩阵 [n_sampled, K] ────────────────────────────────────
print("[2] 计算 influence 分数...")
n_sampled = len(sampled_uids)
influence_overlap    = np.zeros((n_sampled, K), dtype=np.float32)
influence_score_drop = np.zeros((n_sampled, K), dtype=np.float32)

for idx, uid in enumerate(sampled_uids):
    if idx % 100 == 0:
        print(f"  处理用户 {idx}/{n_sampled} ...", end="\r")

    eu = user_emb[uid]                          # (d,)
    scores_before = item_emb @ eu               # (n_items,)  无偏置项
    topk_before   = set(np.argpartition(scores_before, -TOP_K)[-TOP_K:])

    for ki, k in enumerate(hq_patterns):
        vk           = pattern_basis[k]
        eu_probe     = eu - LAMBDA_PROBE * np.dot(eu, vk) * vk
        scores_after = item_emb @ eu_probe
        topk_after   = set(np.argpartition(scores_after, -TOP_K)[-TOP_K:])

        influence_overlap[idx, ki]    = 1.0 - len(topk_before & topk_after) / TOP_K
        I_k = pattern_item_sets[k]
        influence_score_drop[idx, ki] = (scores_before[I_k] - scores_after[I_k]).mean()

print("\n  计算完成")

# ── 3. Min-max 归一化后合并 ───────────────────────────────────────────────────
def minmax(M):
    mn, mx = M.min(), M.max()
    return (M - mn) / (mx - mn + 1e-10)

influence_matrix = 0.5 * minmax(influence_overlap) + 0.5 * minmax(influence_score_drop)
print(f"  influence_matrix shape: {influence_matrix.shape}")
print(f"  均值: {influence_matrix.mean():.4f}，标准差: {influence_matrix.std():.4f}")

# ── 4. 每位用户 influence 从高到低的 (pattern, score) 对 ──────────────────────
print("\n  各用户 influence 排序（前10名用户示例）：")
for idx in range(10):
    uid          = sampled_uids[idx]
    sorted_ki    = np.argsort(influence_matrix[idx])[::-1]
    ranked_pairs = [(hq_patterns[ki], round(float(influence_matrix[idx, ki]), 4))
                    for ki in sorted_ki]
    print(f"  用户 uid={uid}: {ranked_pairs}")

已加载 1000 名用户，首5个uid: [13738, 25771, 9664, 14105, 16776]
[1] 加载数据...
  高质量 pattern 数: 8，物品数: 144853
[2] 计算 influence 分数...
  处理用户 900/1000 ...
  计算完成
  influence_matrix shape: (1000, 8)
  均值: 0.3274，标准差: 0.1070

  各用户 influence 排序（前10名用户示例）：
  用户 uid=13738: [(5, 0.6517), (4, 0.4917), (3, 0.3559), (1, 0.3386), (6, 0.3348), (2, 0.3271), (9, 0.3143), (7, 0.265)]
  用户 uid=25771: [(5, 0.3568), (4, 0.3447), (1, 0.3343), (7, 0.249), (3, 0.2405), (2, 0.2385), (6, 0.215), (9, 0.1974)]
  用户 uid=9664: [(4, 0.4351), (1, 0.3583), (6, 0.2453), (9, 0.2414), (5, 0.2406), (2, 0.2375), (7, 0.2174), (3, 0.216)]
  用户 uid=14105: [(2, 0.5033), (1, 0.4084), (5, 0.3921), (7, 0.3415), (4, 0.3138), (6, 0.3036), (3, 0.25), (9, 0.2437)]
  用户 uid=16776: [(4, 0.7309), (5, 0.4626), (1, 0.3926), (9, 0.3301), (7, 0.321), (2, 0.295), (6, 0.2663), (3, 0.2343)]
  用户 uid=14194: [(4, 0.5108), (1, 0.3734), (5, 0.3569), (9, 0.3536), (2, 0.3459), (3, 0.3167), (6, 0.2989), (7, 0.2162)]
  用户 uid=11562: [(1, 0.6492), (4, 0.4745),

In [10]:
threshold = 0.5

max_influence_per_user = influence_matrix.max(axis=1)  # [1000]
selected_mask = max_influence_per_user >= threshold
selected_indices = np.where(selected_mask)[0]           # 在 sampled_uids 中的位置
selected_uids = [sampled_uids[i] for i in selected_indices]

print(f"阈值: {threshold}")
print(f"满足条件的用户数: {len(selected_uids)} / 1000")
print(f"占比: {len(selected_uids)/10:.1f}%")
print(f"\n最高 influence 分布：")
print(f"  Min    : {max_influence_per_user.min():.4f}")
print(f"  Q1     : {np.percentile(max_influence_per_user, 25):.4f}")
print(f"  Median : {np.median(max_influence_per_user):.4f}")
print(f"  Q3     : {np.percentile(max_influence_per_user, 75):.4f}")
print(f"  Max    : {max_influence_per_user.max():.4f}")
print(f"\n筛选后用户的最高 influence 示例（前10）：")
for i in selected_indices[:10]:
    uid = sampled_uids[i]
    print(f"  uid={uid}: max_influence={max_influence_per_user[i]:.4f}")

阈值: 0.5
满足条件的用户数: 459 / 1000
占比: 45.9%

最高 influence 分布：
  Min    : 0.2359
  Q1     : 0.4090
  Median : 0.4879
  Q3     : 0.5856
  Max    : 0.9585

筛选后用户的最高 influence 示例（前10）：
  uid=13738: max_influence=0.6517
  uid=14105: max_influence=0.5033
  uid=16776: max_influence=0.7309
  uid=14194: max_influence=0.5108
  uid=11562: max_influence=0.6492
  uid=14718: max_influence=0.5027
  uid=16530: max_influence=0.5984
  uid=15741: max_influence=0.5755
  uid=15812: max_influence=0.7038
  uid=15171: max_influence=0.5137


In [15]:
import os

SAVE_DIR     = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/pattern_influence"
UID_PATH     = "/Users/yubinhao/ml-1m/Amazon Review/MF/实验用户id/sampled_uids.npy"
ALPHA_DIR    = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/α与quality"
TOP_K        = 8
LAMBDA_PROBE = 1.0

os.makedirs(SAVE_DIR, exist_ok=True)

# ── 以下变量假设 influence probe 已在同一 session 中运行完毕 ──────────────────
# 直接使用内存中已有的：
#   sampled_uids, hq_patterns, influence_matrix, influence_overlap, influence_score_drop

# 构造每位用户的 ranked (pattern, score) 列表
influence_dict = {}
for idx, uid in enumerate(sampled_uids):
    sorted_ki = np.argsort(influence_matrix[idx])[::-1]
    influence_dict[uid] = [
        (hq_patterns[ki], float(influence_matrix[idx, ki]))
        for ki in sorted_ki
    ]

torch.save(
    {
        "influence_dict":        influence_dict,          # {uid: [(k, score), ...]}
        "influence_matrix":      influence_matrix,        # (n_sampled, K)  归一化后
        "influence_overlap":     influence_overlap,       # (n_sampled, K)  原始
        "influence_score_drop":  influence_score_drop,    # (n_sampled, K)  原始
        "sampled_uids":          sampled_uids,
        "hq_patterns":           hq_patterns,
        "top_k":                 TOP_K,
        "lambda_probe":          LAMBDA_PROBE,
    },
    os.path.join(SAVE_DIR, "influence_scores.pt")
)

print(f"[完成] 保存至 {SAVE_DIR}/influence_scores.pt")
print(f"  用户数: {len(sampled_uids)}, 高质量 pattern 数: {len(hq_patterns)}")
print(f"  influence_matrix: {influence_matrix.shape}, "
      f"mean={influence_matrix.mean():.4f}, std={influence_matrix.std():.4f}")

[完成] 保存至 /Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/pattern_influence/influence_scores.pt
  用户数: 1000, 高质量 pattern 数: 8
  influence_matrix: (1000, 8), mean=0.3274, std=0.1070
